# FairBench: A Dual-Mode Evaluation Framework for Algorithmic Fairness Auditing

**NeurIPS 2026 — Evaluations & Datasets Track — Artifact Showcase**

---

## Abstract

We present **FairBench**, a benchmark and evaluation methodology for comparing automated fairness auditing approaches. FairBench defines a structured Reference Audit Specification (RAS) that formalises what a remediation-ready fairness audit must contain, and measures how well both a deterministic rule-based pipeline and large-language-model (LLM) auditors satisfy that specification on real-world lending and mortgage datasets.

The benchmark makes three contributions:

1. **A project-defined gold standard**: the Reference Audit Specification (RAS), which specifies required metrics, output elements, and quality criteria against which any auditor — deterministic or LLM-based — is scored.
2. **A transparent deterministic baseline**: a fully reproducible Python pipeline that computes five fairness metrics, performs rule-based severity classification and root-cause mapping, retrieves Semantic Scholar evidence, and produces governance recommendations. This pipeline is itself one of the audited systems.
3. **An LLM benchmark protocol**: a multi-cycle qualitative auditing protocol applied to OpenAI `o3` and Google Gemini 2.5 Flash, scored against the RAS using a five-dimension rubric (completeness, severity agreement, cause alignment, mitigation quality, research grounding).

**Evaluative claims this benchmark supports:**
- Which auditing approach better satisfies a pre-specified remediation audit rubric?
- Where do LLM auditors agree or disagree with a deterministic baseline on severity and root cause?
- What is the cost–quality tradeoff between deterministic and LLM auditing?

**What this benchmark does NOT claim:**
- Ground-truth fairness labels (no human audit panel)
- Statistical significance across multiple datasets
- Generalisability beyond the two evaluated datasets

---

**How to use this notebook:**  
Run after `run_pipeline.py` (or with existing `artifacts/`). No API calls are made — all outputs are loaded from saved artifacts. Optional re-run cells are clearly marked.

## §1. Contribution Framing (E&D Track)

### What kind of contribution is this?

This is an **evaluation methodology and benchmark** contribution, not a new fairness algorithm. The central claim is that measuring fairness audit quality requires an explicit, externally-defined specification — not just pairwise model comparison — and that this specification should itself be a versioned, inspectable artefact.

### Role split between systems

| System | Computes metrics? | Produces audit narrative? | Is evaluated against RAS? |
|---|:---:|:---:|:---:|
| Deterministic pipeline | ✓ (Python/AIF360) | ✓ (rule-based) | ✓ |
| OpenAI `o3` | ✗ (receives pre-computed) | ✓ (LLM reasoning) | ✓ |
| Gemini 2.5 Flash | ✗ (receives pre-computed) | ✓ (LLM reasoning) | ✓ |
| Reference Audit Spec | — | — | Acts as gold standard |

### Assumptions

1. The Reference Audit Specification is a project-defined standard, not derived from NIST AI RMF, ISO 42001, or any regulatory body. Its validity is a design assumption, not an empirical claim.
2. The deterministic pipeline's severity thresholds (e.g., DI < 0.80) follow the EEOC 4/5ths rule — a legal heuristic, not a universal fairness law.
3. LLMs receive pre-computed fairness metrics; they do not have access to raw data and cannot independently verify metric values.
4. Semantic Scholar evidence quality is not human-validated; citation count ≥ 5 is used as a relevance proxy.

### Limitations (see §10 for full treatment)

- Two datasets only (German Credit, HMDA Georgia)
- No human evaluation panel for audit quality
- Self-refinement cycles not statistically tested for improvement
- Gemini and OpenAI runs used slightly different protocols (see §6.1)

In [ ]:
# §2. Setup
from pathlib import Path
import json
import textwrap

import pandas as pd
from IPython.display import Markdown, display, Image

# ── Paths ──────────────────────────────────────────────────────────────────
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
ART = REPO / "artifacts"
VIZ = ART / "visualizations"
assert ART.is_dir(), f"Expected artifacts/ at {ART}. Run pipeline first or cd to repo root."
print(f"Repo root : {REPO}")
print(f"Artifacts : {ART}")

# ── Helpers ────────────────────────────────────────────────────────────────
def show_md(path: Path, heading: str = "", max_chars: int = 5000) -> None:
    """Render a (possibly truncated) markdown file."""
    if not path.exists():
        display(Markdown(f"*File not found:* `{path}`"))
        return
    text = path.read_text(encoding="utf-8")
    body = text[:max_chars] + ("\n\n*(truncated — see full file)*" if len(text) > max_chars else "")
    prefix = f"### {heading}\n\n" if heading else ""
    display(Markdown(prefix + body))

def show_img(name: str) -> None:
    p = VIZ / name
    if p.exists():
        display(Image(filename=str(p)))
    else:
        print(f"[missing: {name}]")

def load_json(path: Path) -> dict | list | None:
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))

DATASETS = [
    ("german_credit", "German Credit"),
    ("hmda",          "HMDA Mortgage Lending (Georgia)"),
]
PROVIDERS = [("openai", "OpenAI o3"), ("gemini", "Gemini 2.5 Flash")]

print("Setup complete.")

## §3. The Reference Audit Specification (Gold Standard)

The Reference Audit Specification (RAS) is the project-defined gold standard against which all auditors — deterministic and LLM — are scored. It specifies:

1. **Core metrics** that must be reported: Disparate Impact, Demographic Parity Difference, Equal Opportunity Difference, Average Odds Difference, Theil Index.
2. **Required narrative elements**: per-group breakdown, severity classification, root-cause analysis, mitigation recommendations, research-backed justification.
3. **The rationale**: a remediation-ready audit must couple quantitative disparity measurement with interpretable causes and actionable interventions.

The RAS is embedded in `scripts/llm_benchmark_common.py::REFERENCE_AUDIT_SPEC` and is sent verbatim to all LLM auditors as part of their prompt. It also heads every deterministic qualitative report.

> **Design note:** The RAS is intentionally minimal — it specifies *what* must be in the audit, not *how*. This allows LLMs freedom in narrative style while still being evaluable against a fixed structure.

In [ ]:
# Display the Reference Audit Specification
REFERENCE_AUDIT_SPEC = {
    "name": "Reference Audit Specification",
    "core_metrics": [
        "Disparate Impact",
        "Demographic Parity Difference",
        "Equal Opportunity Difference",
        "Average Odds Difference",
        "Theil Index",
    ],
    "required_response_elements": [
        "per-group breakdown",
        "severity classification",
        "root-cause analysis",
        "mitigation recommendations",
        "research-backed justification",
    ],
    "why_this_spec": (
        "A remediation-ready fairness audit should combine quantitative disparity "
        "measurement with interpretable causes, explicit severity, and concrete "
        "recommended interventions."
    ),
}

spec_md = """| Field | Value |
|---|---|
| Name | {name} |
| Core metrics | {metrics} |
| Required elements | {elements} |
| Rationale | {why} |
""".format(
    name=REFERENCE_AUDIT_SPEC["name"],
    metrics=", ".join(REFERENCE_AUDIT_SPEC["core_metrics"]),
    elements=", ".join(REFERENCE_AUDIT_SPEC["required_response_elements"]),
    why=REFERENCE_AUDIT_SPEC["why_this_spec"],
)
display(Markdown(spec_md))

## §4. Scoring Rubric (Rubric v2)

All auditors are scored on a 100-point scale across five dimensions. **Rubric v2** (used in this notebook) strengthens the mitigation dimension relative to v1, which only keyword-matched algorithm names.

### Point distribution

| Dimension | Max pts | What is measured |
|---|---:|---|
| **Completeness** | 35 | Reference audit spec fields present; all protected attributes covered |
| **Severity agreement** | 20 | LLM severity labels (CRITICAL/HIGH/MODERATE/LOW) match the deterministic baseline |
| **Cause alignment** | 15 | LLM root-cause narrative contains deterministic cause labels |
| **Mitigation quality** | 20 | Algorithm specificity (8) + actionability (6) + governance/monitoring (6) |
| **Research grounding** | 10 | ≥2 citations in spec-level research; ≥1 citation per attribute |
| **Total** | **100** | |

### Rubric v2 vs v1 changes

| Dimension | v1 pts | v2 pts | Change |
|---|---:|---:|---|
| Completeness | 40 | 35 | −5 (redistributed) |
| Mitigation | 15 | 20 | +5 (split into 3 sub-criteria) |
| Others | 45 | 45 | unchanged |

The v1 mitigation score only checked for algorithm name keywords. V2 additionally requires:
- **Actionability**: concrete implementation language (thresholds, parameters, training schedules)
- **Governance**: monitoring, dashboard, compliance, or audit language

This is documented in `scripts/llm_benchmark_common.py::RUBRIC_VERSION`.

## §5. Pipeline Architecture

```
┌─────────────────────────────────────────────────────────────────────────────┐
│  Step 1  Clean → Train → classification_predictions.csv                    │
│           (RandomForest / LogisticRegression per dataset)                   │
│                                    │                                        │
│  Step 2  compute_fairness.py → fairness_metrics.csv                        │
│           AIF360 primary; manual fallback                                   │
│           Metrics: DI, DPD, EOD, AOD, Theil Index                          │
│                                    │                                        │
│  Step 3  qualitative_analysis.py → qualitative_report.md                   │
│  (deterministic)  Rule-based severity → root cause → mitigations           │
│                   + governance recommendations + confidence level           │
│                                    │                                        │
│  Step 4  scholarly_evidence.py → qualitative_research_evidence.json        │
│           Semantic Scholar API; citation-count quality filter               │
│                                    │                                        │
│  Step 5a  openai_fairness_analysis.py  (OpenAI o3, JSON Schema mode)      │
│  Step 5b  llm_fairness_analysis.py     (Gemini 2.5 Flash, free JSON)      │
│           Both receive: pre-computed metrics + per-group breakdown          │
│                         + RAS + Semantic Scholar evidence                  │
│           Output: qualitative audit JSON → scored against RAS               │
│                                    │                                        │
│  Step 6  generate_consolidated_visuals.py → artifacts/visualizations/     │
│           Cross-dataset plots, cycle scores, severity agreement            │
└─────────────────────────────────────────────────────────────────────────────┘
```

**Key architectural property**: LLMs never have access to raw feature data. They receive only Python-computed aggregates. This is a deliberate design choice: it makes the LLM task purely qualitative reasoning, not metric computation, and ensures both deterministic and LLM auditors are compared on the same factual basis.

**Orchestrator**: `run_pipeline.py --datasets german_credit hmda --steps clean train fairness qualitative llm_benchmark visualize`

## §6. Datasets

### 6.1 German Credit (Statlog)

- **Task**: Binary credit approval prediction  
- **Protected attributes**: Sex (male/female), AgeGroup (40+/under 40), foreign_worker (yes/no)  
- **Size**: 1000 records (200 held-out test set)  
- **Known limitation**: 200 test records is small; point estimates have wide confidence intervals, especially for minority subgroups (e.g., foreign workers)

### 6.2 HMDA Mortgage Lending (Georgia)

- **Task**: Binary mortgage loan approval prediction  
- **Protected attributes**: Race (White/non-White), Sex (Male/Female), AgeGroup (mid/other)  
- **Size**: Varies by processed subset; see per-group breakdowns below  
- **Known limitation**: HMDA data only covers one US state (Georgia); results may not generalise

### 6.3 Consolidated Fairness Metrics

In [ ]:
# Load and display consolidated fairness metrics
metrics_path = ART / "consolidated" / "consolidated_fairness_metrics.csv"
if metrics_path.exists():
    df = pd.read_csv(metrics_path)
    # Add severity column
    def classify_severity(di):
        if pd.isna(di): return "UNKNOWN"
        if di < 0.72: return "CRITICAL"
        if di < 0.80: return "HIGH"
        if di < 0.95: return "MODERATE"
        return "LOW (fair)"

    df["Severity"] = df["DisparateImpact"].apply(classify_severity)
    df["BiasFlag"] = df["DisparateImpact"] < 0.80

    # Format for display
    display_df = df[["Dataset", "Attribute", "DisparateImpact", "DemographicParityDiff",
                     "EqualOpportunityDiff", "AverageOddsDiff", "TheilIndex", "Severity"]].copy()
    for col in ["DisparateImpact", "DemographicParityDiff", "EqualOpportunityDiff",
                "AverageOddsDiff", "TheilIndex"]:
        display_df[col] = display_df[col].map(lambda x: f"{x:+.4f}" if pd.notna(x) else "--")
    display_df["DisparateImpact"] = df["DisparateImpact"].map(lambda x: f"{x:.4f}" if pd.notna(x) else "--")

    display(Markdown("**Consolidated fairness metrics across all protected attributes (2 datasets)**"))
    display(display_df.style.set_caption("DI threshold: < 0.80 = biased (EEOC 4/5ths rule)"))
else:
    print(f"Missing: {metrics_path}")

### 6.4 Severity Classification Rules

| Severity | Disparate Impact | Interpretation |
|---|---|---|
| **CRITICAL** | < 0.72 | Severe disparity; immediate intervention required |
| **HIGH** | 0.72 – 0.80 | Clear disparity; mitigation plan needed before deployment |
| **MODERATE** | 0.80 – 0.95 | Borderline; proactive monitoring and root-cause investigation |
| **LOW (fair)** | ≥ 0.95 | Approximately equal treatment; continue monitoring |

**Basis**: The 0.80 threshold follows the EEOC 4/5ths (80%) rule widely adopted in employment law and algorithmic fairness literature. The 0.72 (= 0.80 × 0.90) and 0.95 thresholds are project-defined extensions.

**Limitation**: DI is used as the primary severity driver for simplicity and legal interpretability. In practice, EOD and AOD may be more operationally important depending on the decision context (e.g., a hiring system where missing qualified candidates is the primary harm).

## §7. Deterministic Audit Pipeline Results

The deterministic pipeline (`scripts/qualitative_analysis.py`) is a fully rule-based system with no learned parameters. Every output is a deterministic function of the input metrics, group breakdown, and hardcoded threshold rules. This makes it:

- **Reproducible**: identical inputs always produce identical outputs
- **Inspectable**: every decision can be traced to a specific rule
- **Conservative**: it does not infer causation, only pattern-match to predefined cause templates

The rule engine (`map_root_causes`) checks, in order:
1. Base-rate gap > 10% → historical/label bias
2. Strong/moderate proxy features (r ≥ 0.30) → proxy discrimination
3. Severe group size imbalance → representation bias
4. |EOD| > 0.05 → unequal opportunity
5. |AOD| > 0.05 → unequal odds
6. Borderline DI (0.80–0.95) with no other causes → borderline flag

As of v2, the engine also always emits governance recommendations (monitoring schedule, dashboard alerts, compliance documentation) and a confidence level (HIGH/MEDIUM/LOW based on group sample sizes).

In [ ]:
# §7.1 German Credit — Deterministic audit report (excerpt)
show_md(
    ART / "german_credit" / "fairness" / "qualitative_report.md",
    "Deterministic audit — German Credit",
    max_chars=6000,
)

In [ ]:
# German Credit — per-dataset fairness metrics table
gc_metrics = ART / "german_credit" / "fairness" / "fairness_metrics.csv"
if gc_metrics.exists():
    display(Markdown("**German Credit — fairness metrics by protected attribute**"))
    display(pd.read_csv(gc_metrics))

In [ ]:
# §7.2 HMDA — Deterministic audit report (excerpt)
show_md(
    ART / "hmda" / "fairness" / "qualitative_report.md",
    "Deterministic audit — HMDA",
    max_chars=6000,
)

hmda_metrics = ART / "hmda" / "fairness" / "fairness_metrics.csv"
if hmda_metrics.exists():
    display(Markdown("**HMDA — fairness metrics by protected attribute**"))
    display(pd.read_csv(hmda_metrics))

In [ ]:
# §7.3 Severity and DI visualizations
display(Markdown("#### Disparate Impact — German Credit"))
show_img("german_credit_disparate_impact.png")
display(Markdown("#### Disparate Impact — HMDA"))
show_img("hmda_disparate_impact.png")
display(Markdown("#### Cross-dataset severity comparison"))
show_img("severity_comparison.png")
display(Markdown("#### Metric delta heatmap (DI, DPD, EOD, AOD)"))
show_img("delta_heatmap.png")

In [ ]:
# §7.4 Per-metric breakdowns
for metric in ["demographic_parity_diff", "equal_opportunity_diff",
               "average_odds_diff", "theil_index"]:
    display(Markdown(f"#### {metric.replace('_', ' ').title()}"))
    show_img(f"german_credit_{metric}.png")
    show_img(f"hmda_{metric}.png")

## §8. Semantic Scholar Evidence Retrieval

After the deterministic analysis identifies root causes, `scripts/scholarly_evidence.py` queries the Semantic Scholar Graph API for supporting literature. Key design choices:

| Design choice | Implementation |
|---|---|
| Query strategy | 4 default dataset-level queries + up to 8 attribute-specific queries derived from detected causes and recommended fixes |
| Quality filter | Papers with `citation_count ≥ 5` preferred; newer papers used as fallback |
| Rate limiting | 1.1s between requests; exponential backoff on HTTP 429 |
| Max papers | 6 at dataset level, 3 at attribute level |
| Evidence use | Embedded in deterministic report markdown; sent to LLMs as a compact text block |

**Limitation**: Evidence quality is not human-validated. Citation count is a weak proxy for relevance. Future work should include human-assessed relevance scores.

In [ ]:
# §8.1 Sample evidence packets — German Credit
ev_path = ART / "german_credit" / "fairness" / "qualitative_research_evidence.json"
ev_data = load_json(ev_path)
if ev_data:
    rows = []
    for attr, papers in ev_data.items():
        for p in papers[:3]:
            rows.append({
                "Attribute": attr,
                "Title": p.get("title", "")[:70] + "…",
                "Year": p.get("year"),
                "Citations": p.get("citation_count"),
                "Venue": (p.get("venue") or "")[:30],
                "Query": (p.get("query") or "")[:40],
            })
    display(Markdown("**Sample Semantic Scholar evidence — German Credit**"))
    display(pd.DataFrame(rows))
else:
    print(f"Run qualitative step to generate: {ev_path}")

# HMDA evidence
ev_hmda = ART / "hmda" / "fairness" / "qualitative_research_evidence.json"
ev_hmda_data = load_json(ev_hmda)
if ev_hmda_data:
    rows = []
    for attr, papers in ev_hmda_data.items():
        for p in papers[:2]:
            rows.append({
                "Attribute": attr,
                "Title": p.get("title", "")[:70] + "…",
                "Year": p.get("year"),
                "Citations": p.get("citation_count"),
            })
    display(Markdown("**Sample Semantic Scholar evidence — HMDA**"))
    display(pd.DataFrame(rows))

## §9. LLM Auditor Benchmarking

### 9.1 Benchmark Protocol

Both LLM auditors receive an identical context payload containing:
- Pre-computed fairness metrics (DI, DPD, EOD, AOD, Theil) per attribute
- Per-group breakdowns (n, selection rate, TPR, FPR)
- Severity thresholds
- The Reference Audit Specification
- Semantic Scholar evidence papers
- AIF360 + Fairlearn algorithm catalogue

**LLMs do NOT receive**: raw feature data, ground-truth labels, or the deterministic qualitative narrative.

**Self-refinement**: each LLM runs N=3 cycles. After cycle 1 and 2, the model receives its previous output and a refinement prompt asking it to improve completeness, specificity, and grounding. The score from each cycle is recorded.

**Methodological note on Gemini vs OpenAI comparability**: The OpenAI runs (current) use `response_format=json_schema` with `strict=True` for guaranteed JSON structure. Earlier Gemini runs used free-text JSON with post-hoc parsing. This means raw scores may not be directly comparable across providers; the cycle-score progression and subscore breakdown are more informative than absolute totals.

### 9.2 Cost and Latency Summary

In [ ]:
# Build cost/score summary table from saved artifacts
rows = []
for ds_key, ds_label in DATASETS:
    for prov_key, prov_label in PROVIDERS:
        nap_path = ART / ds_key / "fairness" / prov_key / "llm_napkin_math.json"
        raw_path = ART / ds_key / "fairness" / prov_key / "llm_raw_response.json"
        nap = load_json(nap_path)
        raw = load_json(raw_path)
        if nap is None or raw is None:
            rows.append({"Dataset": ds_label, "Provider": prov_label, "Status": "missing (run llm_benchmark step)"})
            continue
        u = nap.get("usage", nap)
        cycles = raw.get("cycles", [])
        cycle_scores = [c["score"]["total_score"] for c in cycles if c.get("score")]
        rows.append({
            "Dataset": ds_label,
            "Provider": prov_label,
            "Model": u.get("model", "?"),
            "Cycles": len(cycles),
            "Final Score": f"{cycle_scores[-1]:.1f}/100" if cycle_scores else "?",
            "Cycle Gain": f"{cycle_scores[-1] - cycle_scores[0]:+.1f}" if len(cycle_scores) > 1 else "n/a",
            "Cost (USD)": f"${u.get('total_cost_usd', 0):.4f}",
            "Latency (s)": f"{u.get('wall_clock_s', 0):.1f}",
            "Input tokens": u.get("input_tokens"),
            "Output tokens": u.get("output_tokens"),
        })

summary_df = pd.DataFrame(rows)
display(Markdown("**LLM benchmark summary (v1 rubric scores, as stored in artifacts)**"))
display(summary_df)

### 9.3 Enhanced Rubric (v2) Re-Scoring

The stored artifacts contain scores computed with **rubric v1** (mitigation = algorithm keyword matching only, 15 pts; completeness = 40 pts). Here we re-apply **rubric v2** (mitigation quality = 20 pts split into algorithm + actionability + governance; completeness = 35 pts) to the stored LLM outputs.

This illustrates a key benchmark design point: **rubric version is a first-class parameter**. Different rubrics can yield different rankings, and the rubric should be versioned and documented.

In [ ]:
# Apply rubric v2 to stored LLM outputs
# These constants mirror scripts/llm_benchmark_common.py
ALGORITHM_KEYWORDS_V2 = [
    "reweighing", "disparateimpactremover", "prejudiceremover",
    "eqoddspostprocessing", "calibratedeqoddspostprocessing",
    "exponentiatedgradient", "gridsearch", "thresholdoptimizer",
    "equalized odds", "adversarial debiasing", "fairness constraint",
    "demographic parity constraint",
]
ACTIONABILITY_KEYWORDS_V2 = [
    "implement", "apply", "configure", "train with", "deploy",
    "pipeline", "threshold", "parameter", "regularization",
    "class_weight", "retrain", "cross-validation", "held-out",
    "validation set", "schedule", "quarterly", "monthly", "annually",
    "set ", "step ", "target n", "repair_level",
]
GOVERNANCE_KEYWORDS_V2 = [
    "monitor", "audit", "review", "dashboard", "alert", "report",
    "document", "compliance", "governance", "policy", "oversight",
    "periodic", "quarterly", "annual", "log ", "track", "measure",
    "re-audit", "re-evaluate", "drift", "flag", "intersectional",
]

def score_mitigation_quality_v2(result_by_attr: dict) -> dict:
    """Algorithm(8) + Actionability(6) + Governance(6) = 20 pts."""
    if not result_by_attr:
        return {"algorithm": 0, "actionability": 0, "governance": 0, "total": 0}
    algo_hits = action_hits = gov_hits = 0
    n = len(result_by_attr)
    for row in result_by_attr.values():
        raw = (row.get("how_to_fix", "") or "").lower()
        norm = raw.replace("-", "").replace("_", "")
        if any(k.replace("-","").replace("_","") in norm for k in ALGORITHM_KEYWORDS_V2):
            algo_hits += 1
        if any(k in raw for k in ACTIONABILITY_KEYWORDS_V2):
            action_hits += 1
        if any(k in raw for k in GOVERNANCE_KEYWORDS_V2):
            gov_hits += 1
    algo = round(8.0 * algo_hits / n, 2)
    action = round(6.0 * action_hits / n, 2)
    gov = round(6.0 * gov_hits / n, 2)
    return {"algorithm": algo, "actionability": action, "governance": gov, "total": algo + action + gov}

v2_rows = []
for ds_key, ds_label in DATASETS:
    for prov_key, prov_label in PROVIDERS:
        raw_path = ART / ds_key / "fairness" / prov_key / "llm_raw_response.json"
        raw = load_json(raw_path)
        if raw is None:
            continue
        cycles = raw.get("cycles", [])
        if not cycles:
            continue
        final_result = cycles[-1].get("result", {})
        result_by_attr = {r["attribute"]: r for r in final_result.get("qualitative", []) if "attribute" in r}

        # v1 stored score
        v1_score = cycles[-1].get("score", {}).get("total_score", None)
        v1_mit = cycles[-1].get("score", {}).get("subscores", {}).get(
            "mitigation_specificity",
            cycles[-1].get("score", {}).get("subscores", {}).get("mitigation_quality", None)
        )

        # v2 re-score
        mit_v2 = score_mitigation_quality_v2(result_by_attr)
        completeness_delta = -5  # v2 completeness max 35 vs v1 max 40
        mitigation_delta = mit_v2["total"] - (v1_mit or 0)

        v2_rows.append({
            "Dataset": ds_label,
            "Provider": prov_label,
            "v1 Final Score": f"{v1_score:.1f}" if v1_score is not None else "?",
            "v1 Mitigation (max 15)": f"{v1_mit:.1f}" if v1_mit is not None else "?",
            "v2 Algorithm (max 8)": f"{mit_v2['algorithm']:.1f}",
            "v2 Actionability (max 6)": f"{mit_v2['actionability']:.1f}",
            "v2 Governance (max 6)": f"{mit_v2['governance']:.1f}",
            "v2 Mitigation Total (max 20)": f"{mit_v2['total']:.1f}",
        })

if v2_rows:
    display(Markdown("**Rubric v1 vs v2 — mitigation quality comparison**"))
    display(pd.DataFrame(v2_rows))
    display(Markdown(
        "*Note: The v2 re-scoring is applied to the final-cycle LLM outputs as stored in `llm_raw_response.json`. "
        "v1 completeness max was 40; v2 completeness max is 35 (−5 redistributed to mitigation).*"
    ))
else:
    print("No LLM artifacts found — run llm_benchmark step first.")

### 9.4 Severity Agreement Analysis

In [ ]:
# Extract severity labels from stored artifacts
DETERMINISTIC_SEVERITY = {
    # From consolidated_fairness_metrics.csv values
    ("german_credit", "Sex"): "MODERATE",
    ("german_credit", "AgeGroup"): "MODERATE",
    ("german_credit", "foreign_worker"): "LOW (fair)",
    ("hmda", "race"): "MODERATE",
    ("hmda", "sex"): "LOW (fair)",
    ("hmda", "age_group"): "LOW (fair)",
}

sev_rows = []
for ds_key, ds_label in DATASETS:
    for prov_key, prov_label in PROVIDERS:
        raw_path = ART / ds_key / "fairness" / prov_key / "llm_raw_response.json"
        raw = load_json(raw_path)
        if raw is None:
            continue
        cycles = raw.get("cycles", [])
        if not cycles:
            continue
        final_result = cycles[-1].get("result", {})
        for item in final_result.get("qualitative", []):
            attr = item.get("attribute", "")
            llm_sev = (item.get("severity", "") or "").upper()
            det_sev = DETERMINISTIC_SEVERITY.get((ds_key, attr), "UNKNOWN")
            # Normalize
            for label in ["CRITICAL", "HIGH", "MODERATE", "LOW"]:
                if label in llm_sev:
                    llm_sev = label
                    break
            for label in ["CRITICAL", "HIGH", "MODERATE", "LOW"]:
                if label in det_sev:
                    det_sev = label
                    break
            sev_rows.append({
                "Dataset": ds_label,
                "Provider": prov_label,
                "Attribute": attr,
                "Deterministic": det_sev,
                "LLM": llm_sev,
                "Agree": "✓" if det_sev == llm_sev else "✗",
            })

if sev_rows:
    display(Markdown("**Severity agreement: deterministic baseline vs LLM auditors**"))
    display(pd.DataFrame(sev_rows))
else:
    print("No LLM artifacts found.")

### 9.5 LLM Qualitative Reports (Excerpts)

These excerpts show the final-cycle qualitative narratives produced by each LLM auditor. The full reports are at `artifacts/{dataset}/fairness/{provider}/llm_fairness_report.md`.

In [ ]:
# OpenAI — German Credit
show_md(
    ART / "german_credit" / "fairness" / "openai" / "llm_fairness_report.md",
    "OpenAI o3 — German Credit (excerpt)",
    max_chars=4000,
)
# OpenAI — HMDA
show_md(
    ART / "hmda" / "fairness" / "openai" / "llm_fairness_report.md",
    "OpenAI o3 — HMDA (excerpt)",
    max_chars=3500,
)

In [ ]:
# Gemini — German Credit
show_md(
    ART / "german_credit" / "fairness" / "gemini" / "llm_fairness_report.md",
    "Gemini 2.5 Flash — German Credit (excerpt)",
    max_chars=4000,
)
# Gemini — HMDA
show_md(
    ART / "hmda" / "fairness" / "gemini" / "llm_fairness_report.md",
    "Gemini 2.5 Flash — HMDA (excerpt)",
    max_chars=3500,
)

### 9.6 Mitigation Quality Comparison

The following table compares mitigation recommendations across the three systems (deterministic, OpenAI, Gemini) for each dataset × attribute combination. Key patterns to observe:

- **Deterministic**: precise AIF360 algorithm names with library paths; as of v2, includes concrete parameter guidance and governance recommendations
- **OpenAI**: richer operational framing, phased action plans, explicit governance (monitoring dashboards, CI alerts), compliance language
- **Gemini**: varies by run quality; check benchmark comparison artifacts for detailed narrative

In [ ]:
# Build structured mitigation comparison table
det_mitigations = {
    # From deterministic_vs_openai_short_report.md Appendix D
    ("german_credit", "Sex_original"): "Disparate Impact Remover; Prejudice Remover; Equalized Odds post-processing",
    ("german_credit", "AgeGroup_original"): "Reweighing; Disparate Impact Remover; Prejudice Remover; Equalized Odds post-processing",
    ("german_credit", "foreign_worker_original"): "Reweighing; Disparate Impact Remover; more minority-group data; class-weighted training; Prejudice Remover; Equalized Odds post-processing",
    ("hmda", "race"): "Reweighing; collect more underrepresented-group data; class-weighted training",
    ("hmda", "sex"): "Calibrated Equalized Odds post-processing",
    ("hmda", "age_group"): "Reweighing",
}
openai_mitigations = {
    ("german_credit", "Sex_original"): "Reweigh female-positive loans; causal feature pruning; ExponentiatedGradient with demographic parity; calibrated equal-odds hot-fix; CI guardrail on DI",
    ("german_credit", "AgeGroup_original"): "Stratified bootstrap / importance weighting; age-normalized feature ratios; PrejudiceRemover tuning; IRM; age-specific AUC monitoring",
    ("german_credit", "foreign_worker_original"): "Collect more domestic-worker records; SMOTE-NC; bootstrap confidence intervals; group DRO; minimum-support manual-review trigger",
    ("hmda", "race"): "Calibrated Equalized Odds; Fair SMOTE-NC; ExponentiatedGradient; causal feature filtering; race-segmented monitoring dashboard",
    ("hmda", "sex"): "Feature-attribution audit for sex proxies; multi-attribute ExponentiatedGradient; rolling DI operational alert",
    ("hmda", "age_group"): "Keep age on dashboard; recession stress testing for senior FPR; document age-policy rationale with ADEA support",
}

mit_rows = []
for (ds, attr), det in det_mitigations.items():
    mit_rows.append({
        "Dataset": ds.replace("_", " ").title(),
        "Attribute": attr,
        "Deterministic": det[:80] + ("…" if len(det) > 80 else ""),
        "OpenAI o3": (openai_mitigations.get((ds, attr), "—") or "")[:80] + "…",
    })

display(Markdown("**Mitigation comparison: deterministic vs OpenAI**"))
display(pd.DataFrame(mit_rows).set_index(["Dataset", "Attribute"]))

In [ ]:
# Show consolidated comparison report
show_md(
    ART / "consolidated" / "deterministic_vs_openai_short_report.md",
    "Full deterministic vs OpenAI comparison report",
    max_chars=5000,
)

## §10. Cross-Dataset Visualization Suite

All plots are generated by `scripts/generate_consolidated_visuals.py` and `scripts/visualize_openai_benchmark.py`. They are loaded from `artifacts/visualizations/` without recomputation.

In [ ]:
display(Markdown("#### OpenAI self-refinement cycle scores"))
show_img("openai_cycle_scores.png")

display(Markdown("#### OpenAI final subscore breakdown"))
show_img("openai_final_subscores.png")

display(Markdown("#### OpenAI severity agreement with deterministic baseline"))
show_img("openai_severity_agreement.png")

display(Markdown("#### OpenAI cost and latency"))
show_img("openai_cost_latency.png")

display(Markdown("#### DI context: metric values sent to OpenAI"))
show_img("openai_di_context.png")

display(Markdown("#### Cross-provider cost summary"))
show_img("cost_summary.png")

display(Markdown("#### DI overview — all datasets and attributes"))
show_img("di_overview.png")

---

## §11. [PENDING] Multi-Run OpenAI Simulation Results

> **Reserved for teammate-generated simulation outputs.**  
> This section will be populated once the multi-run simulation experiment is complete.  
> Do not modify this section without coordinating with the responsible team member.

### What will go here

This section is reserved for results from **repeated independent runs** of the OpenAI benchmark (and optionally Gemini) on the same datasets, varying:
- Random seed / model temperature
- Number of self-refinement cycles (e.g., N ∈ {1, 2, 3, 5})
- Different prompt phrasings (prompt sensitivity study)

### Expected outputs

Once the simulation outputs are generated and placed in the expected location, the cells below can be uncommented and re-run.

Expected artifact location:
```
artifacts/
└── simulation/
    ├── openai_multi_run_summary.csv       ← per-run scores for all runs
    ├── openai_multi_run_variance.json     ← variance statistics by dataset × attribute
    └── openai_multi_run_plots/            ← violin plots, score distributions
```

### What this section should demonstrate

1. **Score stability**: how much does the final benchmark score vary across runs with the same inputs?
2. **Severity label consistency**: does the LLM always agree with the deterministic baseline, or does it vary?
3. **Mitigation vocabulary consistency**: do the same algorithm names appear across runs, or is the recommendation stochastic?
4. **Cycle gain reliability**: does self-refinement reliably improve scores, or is the gain run-dependent?

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# PENDING: Multi-run simulation results
# To be populated with teammate-generated simulation outputs.
# See §11 markdown cell above for expected artifact structure.
# ══════════════════════════════════════════════════════════════════════

sim_dir = ART / "simulation"
sim_summary = sim_dir / "openai_multi_run_summary.csv"
sim_variance = sim_dir / "openai_multi_run_variance.json"

if sim_summary.exists():
    # ── Uncomment when simulation outputs are available ────────────────
    # sim_df = pd.read_csv(sim_summary)
    # display(Markdown("**Multi-run score summary**"))
    # display(sim_df.describe())
    #
    # if sim_variance.exists():
    #     var_data = load_json(sim_variance)
    #     display(Markdown("**Score variance by dataset**"))
    #     display(pd.DataFrame(var_data))
    pass
else:
    display(Markdown(
        "⏳ **Simulation artifacts not yet available.**  \n"
        "Expected path: `artifacts/simulation/openai_multi_run_summary.csv`  \n"
        "This section is reserved for the teammate running the simulation experiment."
    ))

## §12. [PENDING] Stability and Variance Analysis

> **Reserved for stability / variance analysis.**  
> To be populated with teammate-generated simulation outputs.

### What will go here

Once multi-run simulation results are available (§11), this section will analyse:

1. **Inter-run score variance**: box plots or violin plots of total score distributions per provider × dataset
2. **Severity label stability**: percentage of runs where LLM severity agrees with deterministic baseline
3. **Mitigation vocabulary overlap**: Jaccard similarity of algorithm terms across runs
4. **Statistical significance**: two-sided Wilcoxon signed-rank test for whether cycle self-refinement reliably improves scores (H₀: median improvement = 0)

### Suggested statistical methodology

```python
# Suggested analysis once simulation_df is available
# from scipy import stats
# cycle1_scores = simulation_df["cycle_1_score"]
# cycle3_scores = simulation_df["cycle_3_score"]
# stat, p = stats.wilcoxon(cycle3_scores - cycle1_scores)
# print(f"Wilcoxon p={p:.4f} — {'refinement improves scores' if p < 0.05 else 'no reliable improvement'}")
```

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# PENDING: Stability and variance analysis
# To be populated once §11 simulation outputs exist.
# ══════════════════════════════════════════════════════════════════════

display(Markdown(
    "⏳ **Variance analysis not yet available.**  \n"
    "This section depends on the multi-run simulation outputs from §11.  \n"
    "See the suggested statistical methodology in the §12 markdown cell above."
))

## §13. Strengths, Limitations, Assumptions, and Failure Modes

### Strengths

| Strength | Description |
|---|---|
| **Reproducible baseline** | Deterministic pipeline produces identical outputs for identical inputs; no random parameters |
| **Transparent rule engine** | Every severity label and mitigation recommendation is traceable to a specific rule in `qualitative_analysis.py` |
| **Explicit gold standard** | The Reference Audit Specification is versioned, documented, and sent verbatim to all evaluees |
| **Cost-efficient LLM use** | LLMs only handle qualitative reasoning; metric computation stays in Python. OpenAI cost per dataset: ≈$0.11 |
| **Governance recommendations** | Deterministic pipeline now always emits monitoring schedules, dashboard alerts, and compliance documentation |
| **Multi-dimensional scoring** | Rubric v2 separately evaluates algorithm specificity, actionability, and governance — not just keyword presence |

### Limitations

| Limitation | Impact | Mitigation |
|---|---|---|
| Only 2 datasets | Cannot claim generalisability | Future work: add 3+ datasets across domains |
| No human evaluation panel | Rubric is self-defined; no external validity | Commission expert panel to validate RAS |
| Small test set (German Credit: 200) | Wide confidence intervals on metric estimates | Report bootstrap CIs in future runs |
| Gemini/OpenAI protocol mismatch | Cross-provider comparison is approximate | Unify to same JSON schema constraint |
| Self-refinement not statistically tested | Cycle gain may be noise | Pending §12 variance analysis |
| Evidence not human-validated | Citation count is a weak quality proxy | Add human relevance annotation |
| DI as primary severity driver | Misses cases where EOD or AOD is the primary harm | Add harm-context weighting to severity |
| Theil Index is dataset-level | Reported identically for all attributes | Compute per-attribute generalised entropy |

### Assumptions

1. **EEOC 4/5ths rule**: DI threshold of 0.80 is treated as the primary fairness threshold. This is a legal heuristic, not a universal fairness criterion.
2. **Binary privilege**: each attribute has a single privileged group. Intersectional analysis (e.g., race × sex) is not currently performed.
3. **Pre-computed metrics are correct**: LLMs are assumed to receive accurate metric values from Python. No LLM verification of metric values is possible.
4. **Citation count ≥ 5 as a quality signal**: this is a practical heuristic with known failure modes (e.g., recent high-quality papers not yet widely cited).
5. **Self-defined gold standard**: the RAS is the project's own definition of a good audit; it is not validated against an external authority.

### Failure Modes

| Failure mode | When it occurs | Observable symptom |
|---|---|---|
| Rubric gaming | LLM learns to include keyword terms without meaningful content | High mitigation score but vague recommendations |
| Severity hallucination | LLM assigns CRITICAL to a LOW-DI attribute | Severity disagreement with deterministic baseline |
| Research hallucination | LLM cites papers not in the provided evidence packet | Citations in output not in evidence JSON |
| Proxy feature false positive | Low correlation (r ≈ 0.15) flagged as moderate proxy | Inflated proxy discrimination finding |
| Rule engine false negative | Bias present but not captured by the five cause templates | Missing root cause in deterministic report |
| Refinement cycle regression | Score decreases after self-refinement | Negative cycle gain in score table |

## §14. Conclusions and Recommendations

### Key Findings

1. **Deterministic pipeline as backbone**: The rule-based pipeline is the stronger system for metric computation, severity classification, and reproducibility. It acts as the audit system of record.

2. **LLMs as qualitative second pass**: OpenAI `o3` achieves 85–90/100 on the v1 rubric with full severity agreement (3/3 attributes on both datasets). Its strongest advantage is operational mitigation language: phased action plans, explicit governance, and compliance framing.

3. **Rubric design matters**: The v2 rubric reveals that LLM mitigation quality differs more than the v1 algorithm-keyword score suggests. Actionability and governance scoring distinguish LLMs that produce generic algorithm lists from those that provide implementable plans.

4. **Self-refinement is a signal, not reliable progress**: Cycle gain ranged from −5 to +0 across datasets. Self-refinement should be treated as a quality signal to investigate, not as guaranteed improvement.

5. **Cost-efficiency**: OpenAI `o3` costs ≈$0.11–0.13 per dataset audit — highly cost-effective for the qualitative reasoning it provides, assuming the deterministic metrics are pre-computed.

### Recommendations

| Recommendation | Priority |
|---|---|
| Run the deterministic pipeline as the first-pass audit engine | High |
| Use OpenAI or Gemini as a second-pass qualitative planner | High |
| Commission an expert panel to validate the Reference Audit Specification | High |
| Add bootstrap confidence intervals to all metric point estimates | Medium |
| Unify Gemini and OpenAI to the same JSON schema constraint | Medium |
| Add intersectional fairness analysis (race × sex, etc.) | Medium |
| Add human relevance annotation to Semantic Scholar evidence | Medium |
| Extend to 3+ additional datasets across different domains | Low |

### Recommended Deployment Pattern

```
1. Run deterministic pipeline → system of record for metrics and severity
2. Run Semantic Scholar retrieval → evidence packet
3. Pass metrics + evidence to LLM → operational mitigation plan
4. Human review of LLM recommendations → final audit report
5. Monthly automated DI/DPD monitoring → drift detection
```

---

### File Index

| Artefact | Location |
|---|---|
| Reference Audit Specification | `scripts/llm_benchmark_common.py::REFERENCE_AUDIT_SPEC` |
| Scoring rubric (v2) | `scripts/llm_benchmark_common.py::score_llm_output` |
| Deterministic pipeline | `scripts/qualitative_analysis.py` |
| OpenAI benchmark | `scripts/openai_fairness_analysis.py` |
| Gemini benchmark | `scripts/llm_fairness_analysis.py` |
| Evidence retrieval | `scripts/scholarly_evidence.py` |
| Consolidated metrics | `artifacts/consolidated/consolidated_fairness_metrics.csv` |
| Deterministic reports | `artifacts/{dataset}/fairness/qualitative_report.md` |
| LLM reports | `artifacts/{dataset}/fairness/{provider}/llm_fairness_report.md` |
| Benchmark comparison | `artifacts/consolidated/deterministic_vs_openai_short_report.md` |
| Visualisations | `artifacts/visualizations/*.png` |
| [PENDING] Simulation results | `artifacts/simulation/` (reserved for teammate) |

---

**To re-run the full pipeline:**
```bash
./venv/bin/python run_pipeline.py --datasets german_credit hmda --steps clean train fairness qualitative llm_benchmark visualize
```

Then re-run all cells in this notebook to refresh all figures and tables.